In [1]:
import pandas as pd
import numpy as np

# E-Redes - Data cleaning

Esta notebook muestra el proceso de limpieza  para cada uno de los indicadores seleccionados de E-Redes a partir de las observaciones vistas en el proceso de inspección de datos. Es importante tener en cuenta que solo se van a limpiar datos, cualquier otro proceso relacionado puramente con EDA (p.ej. detectar y gestionar outliers) no forman parte del contexto de esta notebook.

Se tomarán los datasets "raw" de la carpeta 'data_raw/' y, una vez aplicada la limpieza inicial, se guardará una primera versión transformada de cada dataset en la carpeta 'data_interim/' de donde se leerán los datasets de cada indicador limpios con el fin de transformar, combinar y obtener la versión final de los datasets necesarios para pasos posteriores de este proyecto, los cuales se guardarán en 'data_clean/'.

La versión del dataset guardada en 'data_interim/' será en formato 'parquet' dado que permite guardar los datos con los cambios de tipo de datos aplicados, conservando la nueva estructura mejor que en un CSV.

Como varios indicadores utilizan códigos territoriales con formatos distintos, se crea una función global destinada a homogeneizar el código de concelho, que posteriormente actuará como clave para la unión de datasets.

Los códigos territoriales se almacenan como texto, ya que constituyen identificadores y no valores numéricos, permitiendo además conservar los ceros iniciales y gestionar códigos alfanuméricos. El código de concelho está formado por cuatro caracteres: los dos primeros identifican el distrito y los dos siguientes el município. Cuando el dataset contiene códigos DICOFRE de freguesia de seis caracteres, se toman los cuatro primeros caracteres para obtener el código del concelho correspondiente.

Ante la posibilidad de que algunas columnas al ser numéricas hayan eliminado los 0 iniciales, se asegura que todos ellos sean conformados por 4 cifras exactas.

Además, cada dataset nombra esta columna de forma distinta ('codconcelho', 'codigo_concelho', 'con_code', 'cod_concelho'). Al normalizarla, se renombra también a un header común, **'cod_concelho'**, el mismo que se crea en los datasets de INE. Así todas las fuentes comparten la misma clave para el futuro merge y para asignar nombres y regiones desde la tabla de referencia geográfica ('app/utils/concelho_codes/').

In [2]:
def normalize_cod_concelho(series: pd.Series) -> pd.Series:
    return(series.astype("string")
           .str.strip()
           .str.replace(r"[A-Za-z]", "", regex=True)
           .str.zfill(4))

## **8-total-upac-mensal**: Potencia instalada (no obligatoriamente activa) de unidades de autoconsumo

In [3]:
total_upac_df = pd.read_csv(r"../data_raw/8-total-upac-mensal.csv", sep=';', encoding='utf-8')
total_upac_df.head(5)

,data,ano,mes,distrito,concelho,freguesia,codigo_postal,tipo_de_tecnologia,nivel_de_tensao,escalao_de_potencia_instalada,numero_de_instalacoes,potencia_instalada_upac_kw,coddistrito,codconcelho,codfreguesia,cpes,relacao_instalacoes_por_cpe,relacao_potencia_por_cpe
0,2023-10,2023,10,Viseu,Castro daire,Mões,3600,Solar,BTN,"]0, 4]",26.0,42.55,18,1803,180312,12834.0,0.002026,0.003315
1,2023-10,2023,10,Viseu,Castro daire,Monteiras,3600,Solar,BTN,"]0, 4]",5.0,5.75,18,1803,180314,12834.0,0.000390,0.000448
2,2023-10,2023,10,Viseu,Castro daire,"Mamouros, Alva e Ribolhos",3600,Solar,BTN,"]0, 4]",40.0,82.87,18,1803,180323,12834.0,0.003117,0.006457
3,2023-10,2023,10,Viseu,Castro daire,Picão e Ermida,3600,Solar,BTN,"]0, 4]",14.0,21.11,18,1803,180326,12834.0,0.001091,0.001645
4,2023-10,2023,10,Viseu,Castro daire,Reriz e Gafanhão,3600,Solar,BTN,"]0, 4]",10.0,21.11,18,1803,180327,12834.0,0.000779,0.001645


Primero se transforma la columna 'data' a datetime tal como se hizo en la exploración.

In [4]:
total_upac_df['data'] = pd.to_datetime(total_upac_df['data'])
total_upac_df['data'].dtype

dtype('<M8[ns]')

Se renombra la columna de código de concelho al header común 'cod_concelho' y se normaliza con la función global.

In [5]:
total_upac_df = total_upac_df.rename(columns={'codconcelho': 'cod_concelho'}) # Header común para el código de concelho en todos los datasets
total_upac_df['cod_concelho'] = normalize_cod_concelho(total_upac_df['cod_concelho'])
total_upac_df['cod_concelho'].str.len().min(), total_upac_df['cod_concelho'].str.len().max()

(np.int64(4), np.int64(4))

Se categorizan las columnas tipo_de_tecnologia, nivel_de_tensao y escalao_de_potencia_instalada.
Las dos últimas requieren de un orden. Para determinar todas las categorías que existen y evitar tener errores tipográficos podemos usar '.value_counts()', también serviría '.unique()', si bien con el recuento podemos observar inconsistencias.

In [6]:
total_upac_df['tipo_de_tecnologia'].value_counts()

tipo_de_tecnologia
Solar                      465922
Não Atribuído               23944
Eólica                        544
Biogás                        173
Hídrica                        57
Cogeração não renovável        42
Fotovoltaica                   30
Biomassa                       24
Cogeração NFER                  2
Name: count, dtype: int64

In [7]:
total_upac_df['nivel_de_tensao'].value_counts()

nivel_de_tensao
BTN    300385
BTE     94455
MT      94312
AT       1592
Name: count, dtype: int64

In [8]:
total_upac_df['escalao_de_potencia_instalada'].value_counts()

escalao_de_potencia_instalada
]0, 4]        166537
]4, 20.7]     148149
]30, 1000]     94470
]20.7, 30]     80817
>1000            762
ND                24
Name: count, dtype: int64

In [9]:
nivel_tensao_order = ['BTN', 'BTE', 'MT', 'AT'] # Baja tensión normal, baja tensión especial, media tensión, alta tensión

In [10]:
escalao_potencia_order = [
    ']0, 4]',
    ']4, 20.7]',
    ']20.7, 30]',
    ']30, 1000]',
    '>1000'
]

En cuanto a la escala de potencia instalada; hay que tratar 'ND' como NaN. Si bien la categorización ya los transforma a NaN al no incluirse como categoría, es más limpio si se establece de entrada.

Lo mismo con 'Não Atribuído' de la columna 'tipo_de_potencia' si bien se hará justificadamente más adelante.

In [11]:
total_upac_df['escalao_de_potencia_instalada'] = total_upac_df['escalao_de_potencia_instalada'].replace('ND', pd.NA)

In [12]:
total_upac_df['tipo_de_tecnologia'] = total_upac_df['tipo_de_tecnologia'].astype("category")

total_upac_df['nivel_de_tensao'] = pd.Categorical(total_upac_df['nivel_de_tensao'], categories=nivel_tensao_order, ordered=True)

total_upac_df['escalao_de_potencia_instalada'] = pd.Categorical(total_upac_df['escalao_de_potencia_instalada'], categories=escalao_potencia_order, ordered=True)

In [13]:
total_upac_df['tipo_de_tecnologia'].dtype, total_upac_df['nivel_de_tensao'].dtype, total_upac_df['escalao_de_potencia_instalada'].dtype

(CategoricalDtype(categories=['Biogás', 'Biomassa', 'Cogeração NFER',
                   'Cogeração não renovável', 'Eólica', 'Fotovoltaica',
                   'Hídrica', 'Não Atribuído', 'Solar'],
 , ordered=False, categories_dtype=object),
 CategoricalDtype(categories=['BTN', 'BTE', 'MT', 'AT'], ordered=True, categories_dtype=object),
 CategoricalDtype(categories=[']0, 4]', ']4, 20.7]', ']20.7, 30]', ']30, 1000]', '>1000'], ordered=True, categories_dtype=object))

In [14]:
cols = [
    "distrito",
    "concelho",
    "cpes",
    "relacao_instalacoes_por_cpe",
    "relacao_potencia_por_cpe"
]

total_upac_df[cols].isna().sum()

distrito                       1511
concelho                       1511
cpes                           1511
relacao_instalacoes_por_cpe    1511
relacao_potencia_por_cpe       1511
dtype: int64

In [15]:
total_upac_df[
    total_upac_df["distrito"].isna()
][cols].isna().sum()

distrito                       1511
concelho                       1511
cpes                           1511
relacao_instalacoes_por_cpe    1511
relacao_potencia_por_cpe       1511
dtype: int64

In [16]:
(
    total_upac_df['distrito'].isna()
    == total_upac_df['concelho'].isna()
).all()

np.True_

Vemos que encajan los NaN de todas las variables, esto implica que se sitúan en las mismas filas.

Si bien en lo que respecta a distrito no hay una necesidad de imputación, sí que podríamos imputar los nombres de los concelhos aprovechando que tenemos todos los códigos de concelho sin NaN.

Para ello lo primero es asegurarse de que cada código de concelho apunta a un solo valor de concelho.

In [17]:
total_upac_df.groupby('cod_concelho')['concelho'].nunique().min(), total_upac_df.groupby('cod_concelho')['concelho'].nunique().max()

(np.int64(1), np.int64(1))

In [18]:
total_upac_df.groupby('concelho')['cod_concelho'].nunique().max()

np.int64(1)

Queda comprobado que cada código de concelho tiene un nombre de concelho y viceversa.

Se procede a crear un mapa que permita asociar códigos con nombres de concelho.

In [19]:
cod_concelho_map = total_upac_df.dropna(subset=['concelho']).drop_duplicates(subset=['cod_concelho']).set_index('cod_concelho')['concelho']
cod_concelho_map.head(5)

cod_concelho
1803         Castro daire
1804              Cinfães
1805               Lamego
1806            Mangualde
1807    Moimenta da Beira
Name: concelho, dtype: object

In [20]:
col_mapped = total_upac_df['cod_concelho'].map(cod_concelho_map)

total_upac_df['concelho'] = total_upac_df['concelho'].fillna(col_mapped)

In [21]:
total_upac_df['concelho'].isna().sum()

np.int64(0)

En lo que respecta a los NaN de distrito, se ha comentado que no se tratarán.

La columna cpes hace referencia al Código do Ponto de Entrega, un identificador único, digital e invariable de 20 dígitos que la distribuidora asigna a cada punto de suministro de energía. En este proyecto no tendrá aplicación útil por lo que no se gestionan sus NaN.

Esto afecta directamente a 'relacao_instalacoes_por_cpe' y a 'relacao_potencia_por_cpe'; dado que trabajaremos a nivel de concelho.

Los NaN restantes son relativos al tipo de tecnología y nivel de tensión, hay apenas 21 y 15 respectivamente en un dataset de alrededor de 500k registros.

Primero observamos los registros únicos que muestran NaN en su tipo de tecnología. Mostraremos algunas de sus dimensiones.

In [22]:
missing_tech = total_upac_df[
    total_upac_df["tipo_de_tecnologia"].isna()
]

missing_tech[
    [
        "codfreguesia",
        "potencia_instalada_upac_kw",
        "nivel_de_tensao"
    ]
].drop_duplicates()

,codfreguesia,potencia_instalada_upac_kw,nivel_de_tensao
40587,131311,249.61,MT
91191,011106,1428.00,MT
91192,090753,1320.00,MT
91193,131501,1654.70,MT
153892,060220,2000.00,AT
227997,100936,1760.00,MT


Ahora se toma el código de freguesia y la potencia instalada de uno de los casos con valores NaN para filtrar el dataset entero mostrando todas las coincidencias con estos mismos dos datos que existen, y mostrando además el tipo de tecnología para ver si se trata de la misma instalación para la cual en algunos periodos de tiempo no se registró bien el tipo de tecnologia.

In [23]:
total_upac_df[
    (total_upac_df["codfreguesia"] == "011106") &
    (total_upac_df["potencia_instalada_upac_kw"] == 1428)
][
    [
        "data",
        "tipo_de_tecnologia",
        "nivel_de_tensao",
        "potencia_instalada_upac_kw"
    ]
].sort_values("data")

,data,tipo_de_tecnologia,nivel_de_tensao,potencia_instalada_upac_kw
102127,2025-03-01,NaN,MT,1428.0
274620,2025-04-01,NaN,MT,1428.0
456402,2025-05-01,NaN,MT,1428.0
91191,2025-06-01,NaN,MT,1428.0
126690,2025-07-01,Solar,MT,1428.0
116133,2025-08-01,Solar,MT,1428.0
178062,2025-09-01,Solar,MT,1428.0
421541,2025-10-01,Solar,MT,1428.0
419626,2025-11-01,Solar,MT,1428.0
123622,2025-12-01,Solar,MT,1428.0


Se observa como evidentemente parece que para este primer caso los NaN son de forma muy segura: 'Solar'.

De todas formas sería interesante comprobar si para cada combinación única de código de freguesia, potencia instalada y nivel de tensión, existe un único tipo de tipo de tecnología, en cuyo caso sería posible imputar dicho tipo a los NaN existentes para cada grupo.

Primero se "limpia" el dataset eliminando primero cualquier duplicado que comparta columnas clave (incluyendo el tipo de tecnología), luego eliminando cualquier registro que contenga 'Não Atribuído' como valor para tipo de tecnología oque directamente tenga NaN.

La idea es dejar combinaciones únicas de ['codfreguesia', 'nivel_de_tensao', 'potencia_instalada_upac_kw', 'tipo_de_tecnologia'] para posteriormente comprobar si existen casos con más de un tipo de tecnología para un mismo conjunto de ['codfreguesia', 'nivel_de_tensao', 'potencia_instalada_upac_kw'].

In [24]:
clean_tecnologia = (total_upac_df[['codfreguesia', 'nivel_de_tensao', 'potencia_instalada_upac_kw', 'tipo_de_tecnologia']]
                    .dropna(subset='tipo_de_tecnologia')
                    .query("tipo_de_tecnologia != 'Não Atribuído'")
                    .drop_duplicates(subset=['codfreguesia', 'nivel_de_tensao', 'potencia_instalada_upac_kw', 'tipo_de_tecnologia']))

In [25]:
clean_tecnologia[
    clean_tecnologia.duplicated(subset=['codfreguesia', 'nivel_de_tensao', 'potencia_instalada_upac_kw'], keep=False)
]

,codfreguesia,nivel_de_tensao,potencia_instalada_upac_kw,tipo_de_tecnologia
16588,181005,MT,2100.0,Eólica
264880,181005,MT,2100.0,Solar


Queda comprobado que para una misma combinación de código de freguesia, nivel de tensión y potencia instalada hay solo un tipo de tecnologia asociado a todas las combinaciones existentes menos una, presumiblemente se trata de un registro erróneo que habrá que comprobar.

Esto significa que se puede crear un 'map' que permita asociar un tipo de tecnología a cada combinación concreta de columnas clave para posteriormente aplicarlo en el conjunto del dataset.

Antes se gestiona la situación del único duplicado obtenido en la comprobación anterior.

In [26]:
total_upac_df[(total_upac_df['codfreguesia'] == '181005') & (total_upac_df['potencia_instalada_upac_kw'] == 2100.0)].sort_values(by='data')

,data,ano,mes,distrito,concelho,freguesia,codigo_postal,tipo_de_tecnologia,nivel_de_tensao,escalao_de_potencia_instalada,numero_de_instalacoes,potencia_instalada_upac_kw,coddistrito,cod_concelho,codfreguesia,cpes,relacao_instalacoes_por_cpe,relacao_potencia_por_cpe
400852,2024-04-01,2024,4,Viseu,Oliveira de Frades,Pinheiro,3680,Solar,MT,>1000,1.0,2100.0,18,1810,181005,6927.0,0.000144,0.303162
311132,2024-05-01,2024,5,Viseu,Oliveira de Frades,Pinheiro,3680,Solar,MT,>1000,1.0,2100.0,18,1810,181005,6927.0,0.000144,0.303162
264880,2024-06-01,2024,6,Viseu,Oliveira de Frades,Pinheiro,3680,Solar,MT,>1000,1.0,2100.0,18,1810,181005,6927.0,0.000144,0.303162
355931,2024-07-01,2024,7,Viseu,Oliveira de Frades,Pinheiro,3680,Solar,MT,>1000,1.0,2100.0,18,1810,181005,6927.0,0.000144,0.303162
16588,2024-08-01,2024,8,Viseu,Oliveira de Frades,Pinheiro,3680,Eólica,MT,>1000,1.0,2100.0,18,1810,181005,6927.0,0.000144,0.303162
33127,2024-09-01,2024,9,Viseu,Oliveira de Frades,Pinheiro,3680,Eólica,MT,>1000,1.0,2100.0,18,1810,181005,6927.0,0.000144,0.303162
366985,2024-10-01,2024,10,Viseu,Oliveira de Frades,Pinheiro,3680,Eólica,MT,>1000,1.0,2100.0,18,1810,181005,6927.0,0.000144,0.303162
328438,2024-11-01,2024,11,Viseu,Oliveira de Frades,Pinheiro,3680,Eólica,MT,>1000,1.0,2100.0,18,1810,181005,6927.0,0.000144,0.303162
216309,2024-12-01,2024,12,Viseu,Oliveira de Frades,Pinheiro,3680,Eólica,MT,>1000,1.0,2100.0,18,1810,181005,6927.0,0.000144,0.303162
127731,2025-01-01,2025,1,Viseu,Oliveira de Frades,Pinheiro,3680,Eólica,MT,>1000,1.0,2100.0,18,1810,181005,6927.0,0.000144,0.303162


Parece que simplemente hubo un cambio de tipo de tecnología en Agosto de 2024, además, no se observa ningún NaN para este subconjunto de datos por lo que no se verá afectado por el valor que se le asigne, siendo seguro proceder con el "mapping" para imputar los NaN de 'tipo_de_tecnologia'.

Al hacer el mapa, una vez eliminados los registros con NaN en tipo de tecnología y quitados los registros con 'Não Atribuído' en esa misma columna, se eliminan duplicados dejando únicamente el último registro, de manera que el tipo de tecnología que se tomará como global para el grupo de columnas clave pertinente, será el último del que haya constancia, en el caso anterior sería 'Eólica' si bien no tiene influencia en el resultado final.

In [27]:
tech_map = total_upac_df.dropna(subset='tipo_de_tecnologia').query("tipo_de_tecnologia != 'Não Atribuído'").drop_duplicates(subset=['codfreguesia', 'nivel_de_tensao', 'potencia_instalada_upac_kw'], keep='last').set_index(keys=['codfreguesia', 'nivel_de_tensao', 'potencia_instalada_upac_kw'])['tipo_de_tecnologia']

tech_map.head(5)

codfreguesia  nivel_de_tensao  potencia_instalada_upac_kw
180312        BTN              42.55                         Solar
180323        BTN              82.87                         Solar
180505        BTN              23.57                         Solar
180507        BTN              28.61                         Solar
180527        BTN              37.34                         Solar
Name: tipo_de_tecnologia, dtype: category
Categories (9, object): ['Biogás', 'Biomassa', 'Cogeração NFER', 'Cogeração não renovável', ..., 'Fotovoltaica', 'Hídrica', 'Não Atribuído', 'Solar']

El mapa consiste en una Series con MultiIndex formado por las tres columnas claves y con el tipo de tecnología como valor.

Aplicando el mapa sobre el dataset entero se obtiene un Index con los valores de tipo de tecnología asociados a la combinación de columnas clave encontradas para cada registro, dicho lo cual, habrá que convertir este Index a Series para poder aplicar correctamente la función '.fillna()'.

Para aplicar el 'mapping' habrá que establecer el mismo MultiIndex sobre el dataset entero, luego extraerlo y aplicar el 'mapping'.

In [28]:
mapped_tech = pd.Series(total_upac_df.set_index(['codfreguesia', 'nivel_de_tensao', 'potencia_instalada_upac_kw']).index.map(tech_map), index=total_upac_df.index)
mapped_tech

0         Solar
1         Solar
2         Solar
3         Solar
4         Solar
          ...  
490754      NaN
490755    Solar
490756      NaN
490757      NaN
490758    Solar
Length: 490759, dtype: category
Categories (9, object): ['Biogás', 'Biomassa', 'Cogeração NFER', 'Cogeração não renovável', ..., 'Fotovoltaica', 'Hídrica', 'Não Atribuído', 'Solar']

Parece que en la Series resultante tras el 'mapping' existen 'NaN' para algunos registros, seguramente sean registros cuya agrupación de columnas clave (codfreguesia + nivel de tensión + potencia instalada) presentaba únicamente valores 'NaN' y 'Não Atribuído' para la columna 'tipo_de_tecnologia', de forma que al crear el mapa directamente quedaban descartadas.

De todas formas, ejecutaremos '.fillna()' para imputar valores NaN sobre la dimensión 'tipo_de_tecnologia' y comprobaremos si el procedimiento ha reducido considerablemente el número de NaN para dicha columna.
Para asegurarnos de que el proceso ha sido consistente, comprobaremos que se confirma la suposición hecha en este mismo bloque sobre los NaN que siguen existiendo.

In [29]:
total_upac_df['tipo_de_tecnologia'] = total_upac_df['tipo_de_tecnologia'].fillna(mapped_tech)
total_upac_df['tipo_de_tecnologia'].isna().sum()

np.int64(16)

Se ha reducido el número de valores 'missing' en 5 unidades. Se comprueban los NaN restantes.

In [30]:
total_upac_df[total_upac_df['tipo_de_tecnologia'].isna()].drop_duplicates(subset=['codfreguesia', 'nivel_de_tensao', 'potencia_instalada_upac_kw'])

,data,ano,mes,distrito,concelho,freguesia,codigo_postal,tipo_de_tecnologia,nivel_de_tensao,escalao_de_potencia_instalada,numero_de_instalacoes,potencia_instalada_upac_kw,coddistrito,cod_concelho,codfreguesia,cpes,relacao_instalacoes_por_cpe,relacao_potencia_por_cpe
40587,2025-04-01,2025,4,Porto,Póvoa de Varzim,Rates,4570,NaN,MT,"]30, 1000]",1.0,249.61,13,1313,131311,46169.0,0.000022,0.005406
91193,2025-06-01,2025,6,Porto,Valongo,Alfena,4440,NaN,MT,>1000,1.0,1654.70,13,1315,131501,49107.0,0.000020,0.033696
153892,2025-04-01,2025,4,Coimbra,Cantanhede,Cantanhede e Pocariça,3060,NaN,AT,>1000,1.0,2000.00,06,0602,060220,24713.0,0.000040,0.080929
227997,2025-03-01,2025,3,Leiria,Leiria,Monte Redondo e Carreira,2425,NaN,MT,>1000,1.0,1760.00,10,1009,100936,85387.0,0.000012,0.020612


In [31]:
total_upac_df[(total_upac_df['codfreguesia']=='060220') & (total_upac_df['potencia_instalada_upac_kw']==2000.00)]

,data,ano,mes,distrito,concelho,freguesia,codigo_postal,tipo_de_tecnologia,nivel_de_tensao,escalao_de_potencia_instalada,numero_de_instalacoes,potencia_instalada_upac_kw,coddistrito,cod_concelho,codfreguesia,cpes,relacao_instalacoes_por_cpe,relacao_potencia_por_cpe
52063,2025-09-01,2025,9,Coimbra,Cantanhede,Cantanhede e Pocariça,3060,Não Atribuído,AT,>1000,1.0,2000.0,06,0602,060220,24713.0,0.00004,0.080929
60535,2025-10-01,2025,10,Coimbra,Cantanhede,Cantanhede e Pocariça,3060,Não Atribuído,AT,>1000,1.0,2000.0,06,0602,060220,24713.0,0.00004,0.080929
64153,2026-01-01,2026,1,Coimbra,Cantanhede,Cantanhede e Pocariça,3060,Não Atribuído,AT,>1000,1.0,2000.0,06,0602,060220,24713.0,0.00004,0.080929
105421,2025-11-01,2025,11,Coimbra,Cantanhede,Cantanhede e Pocariça,3060,Não Atribuído,AT,>1000,1.0,2000.0,06,0602,060220,24713.0,0.00004,0.080929
126693,2025-07-01,2025,7,Coimbra,Cantanhede,Cantanhede e Pocariça,3060,Não Atribuído,AT,>1000,1.0,2000.0,06,0602,060220,24713.0,0.00004,0.080929
153892,2025-04-01,2025,4,Coimbra,Cantanhede,Cantanhede e Pocariça,3060,NaN,AT,>1000,1.0,2000.0,06,0602,060220,24713.0,0.00004,0.080929
227995,2025-03-01,2025,3,Coimbra,Cantanhede,Cantanhede e Pocariça,3060,NaN,AT,>1000,1.0,2000.0,06,0602,060220,24713.0,0.00004,0.080929
241453,2025-12-01,2025,12,Coimbra,Cantanhede,Cantanhede e Pocariça,3060,Não Atribuído,AT,>1000,1.0,2000.0,06,0602,060220,24713.0,0.00004,0.080929
277189,2026-04-01,2026,4,Coimbra,Cantanhede,Cantanhede e Pocariça,3060,Não Atribuído,AT,>1000,1.0,2000.0,06,0602,060220,24713.0,0.00004,0.080929
336843,2026-02-01,2026,2,Coimbra,Cantanhede,Cantanhede e Pocariça,3060,Não Atribuído,AT,>1000,1.0,2000.0,06,0602,060220,24713.0,0.00004,0.080929


Efectivamente, los NaN restantes para la columna 'tipo_de_tecnologia' se deben por pertenecer a grupos ['codfreguesia', 'nivel_de_tensao', 'potencia_instalada_upac_kw'] en los que sus valores 'tipo_de_tecnología' asociados son o bien NaN o bien 'Não Atribuído'.
Por esa razón, se sustituirán los 'Não Atribuído' con pd.NA, dado que conceptualmente tienen el mismo peso y con los missing de pandas se puede trabajar mucho mejor durante el análisis posterior (funciones propias para gestión de missings).

In [32]:
total_upac_df['tipo_de_tecnologia'] = total_upac_df['tipo_de_tecnologia'].replace('Não Atribuído', pd.NA)
total_upac_df['tipo_de_tecnologia'].isna().sum(), (total_upac_df['tipo_de_tecnologia'].isna().mean() * 100).round(2)

C:\Users\roger\AppData\Local\Temp\ipykernel_4468\4275079256.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  total_upac_df['tipo_de_tecnologia'] = total_upac_df['tipo_de_tecnologia'].replace('Não Atribuído', pd.NA)


(np.int64(23960), np.float64(4.88))

Convertimos a tipo categórico tras el tratamiento de NaN.

In [33]:
total_upac_df['tipo_de_tecnologia'] = (
    total_upac_df['tipo_de_tecnologia']
    .astype('category')
)

Ahora tenemos realmente todos los NaN que estaban escondidos bajo la categoría 'Não Atribuído', los cuales representan unn 4.88% del total.

El procedimiento debería ser el mismo para 'nivel_de_tensao'. Se hacen las comprobaciones necesarias.

In [34]:
total_upac_df[total_upac_df['nivel_de_tensao'].isna()][['nivel_de_tensao','codfreguesia', 'tipo_de_tecnologia', 'potencia_instalada_upac_kw']]

,nivel_de_tensao,codfreguesia,tipo_de_tecnologia,potencia_instalada_upac_kw
14814,NaN,110507,Solar,6.12
14862,NaN,111013,Solar,10.20
14881,NaN,111128,Solar,13.32
16587,NaN,151301,Solar,5460.00
23400,NaN,110507,Solar,6.12
23467,NaN,111128,Solar,13.32
201351,NaN,110501,Solar,10.00
201373,NaN,110658,Solar,8.00
201401,NaN,111014,Solar,10.20
203259,NaN,040721,Solar,22.00


In [35]:
total_upac_df[(total_upac_df['codfreguesia'] == '110507') & (total_upac_df['potencia_instalada_upac_kw'] == 6.12)][
    [
        "data",
        "tipo_de_tecnologia",
        "nivel_de_tensao",
        "potencia_instalada_upac_kw"
    ]
]

,data,tipo_de_tecnologia,nivel_de_tensao,potencia_instalada_upac_kw
14814,2024-04-01,Solar,NaN,6.12
23400,2024-05-01,Solar,NaN,6.12


Una vez comprobada la existencia de grupos en las que no hay un nivel de tensión para tomar como referencia clara, se procede a valorar la adjudicación del nivel de tensión a partir de la potencia instalada como "frontera" de nivel.
Para ello, se revisa si existen dichas fronteras de potencia.

In [36]:
total_upac_df.groupby("nivel_de_tensao")[
    "potencia_instalada_upac_kw"
].agg(["min", "max"])

C:\Users\roger\AppData\Local\Temp\ipykernel_4468\768204582.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  total_upac_df.groupby("nivel_de_tensao")[


,min,max
nivel_de_tensao,,
BTN,0.00,4321.46
BTE,0.00,1509.23
MT,0.25,13896.48
AT,0.00,19600.00


Queda descartada, de modo que estos 15 NaN quedan sin imputar. Tampoco se eliminarán, se tendrá en cuenta si durante el proceso de EDA se necesita trabajar con dicha variable.

Cabe comprobar la consistencia textual para evitar casos en los que una variable presente un mismo valor con diferencias mínimas por errores o por falta de homogeneidad (mayúsculas y minúsculas mezcladas).

In [37]:
total_upac_df['tipo_de_tecnologia'].value_counts(dropna=False)

tipo_de_tecnologia
Solar                      465927
NaN                         23960
Eólica                        544
Biogás                        173
Hídrica                        57
Cogeração não renovável        42
Fotovoltaica                   30
Biomassa                       24
Cogeração NFER                  2
Name: count, dtype: int64

In [38]:
total_upac_df['nivel_de_tensao'].value_counts(dropna=False)

nivel_de_tensao
BTN    300385
BTE     94455
MT      94312
AT       1592
NaN        15
Name: count, dtype: int64

In [39]:
total_upac_df['escalao_de_potencia_instalada'].value_counts(dropna=False)

escalao_de_potencia_instalada
]0, 4]        166537
]4, 20.7]     148149
]30, 1000]     94470
]20.7, 30]     80817
>1000            762
NaN               24
Name: count, dtype: int64

Para detectar posibles inconsistencias de escritura en variables categóricas con muchos valores, se crea temporalmente una versión normalizada del texto, eliminando espacios al inicio/final y convirtiendo a minúsculas.

Por ejemplo:

```text
freguesia      normalized
Lisboa         lisboa
 lisboa        lisboa
LISBOA         lisboa
Porto          porto
```

Después se agrupa por el valor normalizado y se comprueba cuántas formas originales distintas existen para cada grupo. Si un mismo valor normalizado presenta más de una forma original, puede indicar una inconsistencia de escritura que conviene revisar.


In [40]:
distrito_check = (
    total_upac_df
    .dropna(subset=['distrito'])
    .assign(normalized=total_upac_df['distrito'].astype('string').str.strip().str.lower())
    .groupby('normalized')['distrito']
    .nunique()
)

distrito_check[distrito_check > 1]

Series([], Name: distrito, dtype: int64)

In [41]:
concelho_check = (
    total_upac_df
    .dropna(subset=['concelho'])
    .assign(normalized=total_upac_df['concelho'].astype('string').str.strip().str.lower())
    .groupby('normalized')['concelho']
    .nunique()
)

concelho_check[concelho_check > 1]

Series([], Name: concelho, dtype: int64)

In [42]:
freguesia_check = (
    total_upac_df
    .dropna(subset=['freguesia'])
    .assign(normalized=total_upac_df['freguesia'].astype('string').str.strip().str.lower())
    .groupby('normalized')['freguesia']
    .nunique()
)

freguesia_check[freguesia_check > 1]

Series([], Name: freguesia, dtype: int64)

In [43]:
total_upac_df.groupby('coddistrito')['distrito'].nunique().sort_values(ascending=False).max()

np.int64(1)

In [44]:
total_upac_df.groupby('cod_concelho')['concelho'].nunique().sort_values(ascending=False).max()

np.int64(1)

In [45]:
total_upac_df.groupby('codfreguesia')['freguesia'].nunique().sort_values(ascending=False).max()

np.int64(1)

**Nota sobre los nombres territoriales.** Las comprobaciones anteriores verifican que, dentro de este dataset, cada código (distrito, concelho, freguesia) corresponde a un único nombre y viceversa, y que los nombres no tienen variantes de escritura. Sirven para validar la calidad de la fuente y, en este caso, han permitido imputar nombres de concelho a partir de su código.

Sin embargo, los nombres de cada dataset no serán los definitivos: al construir el dataset conjunto (tras el EDA univariado), el nombre de concelho, distrito o región NUTS se asignará desde la tabla de referencia geográfica ('app/utils/concelho_codes/concelho_reference.csv') a partir de 'cod_concelho'. De este modo todas las fuentes compartirán la misma nomenclatura aunque escriban los nombres de forma distinta.

No se requiere comprobación de valores numéricos incorrectos dado que en la fase de exploración se han descartado.

Comprobación final del dataset entero:

In [46]:
total_upac_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 490759 entries, 0 to 490758
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   data                           490759 non-null  datetime64[ns]
 1   ano                            490759 non-null  int64         
 2   mes                            490759 non-null  int64         
 3   distrito                       489248 non-null  object        
 4   concelho                       490759 non-null  object        
 5   freguesia                      489001 non-null  object        
 6   codigo_postal                  490759 non-null  int64         
 7   tipo_de_tecnologia             466799 non-null  category      
 8   nivel_de_tensao                490744 non-null  category      
 9   escalao_de_potencia_instalada  490735 non-null  category      
 10  numero_de_instalacoes          490759 non-null  float64       
 11  

Se guarda el dataset modificado en formato 'Parquet', puesto que permite conservar mejor la estructura y tipos del DataFrame en comparación a los CSV.
Para los datos guardados en 'data_interim/' este será el formato estándar.

In [47]:
from pathlib import Path
Path("../data_interim").mkdir(parents=True, exist_ok=True)

In [48]:
total_upac_df.to_parquet(r"../data_interim/total_upac_interim.parquet", index=False)

## **energia_injectada_upac**: Energía inyectada a la red por instalaciones de autoconsumo (activas)

In [49]:
energia_injectada_df = pd.read_csv(r"../data_raw/energia_injectada_upac.csv", sep=';', encoding='utf-8')
energia_injectada_df.head(5)

,data,ano,mes,ano_mes,distrito,con_name,nivel_tensao,potencia_instalada,numero_de_instalacoes,energia,codigo_distrito,codigo_concelho,codigo_freguesia
0,2025-10,2025,10,202510,Viana do Castelo,Vila Nova de Cerveira,BTN/BTE,980.48,362,34379.180,16,1610,161000
1,2025-06,2025,6,202506,Viseu,Vouzela,BTN/BTE,2183.14,607,134390.782,18,1824,182400
2,2025-12,2025,12,202512,Portalegre,Elvas,MT/AT,4215.74,40,71816.250,12,1207,120700
3,2025-05,2025,5,202505,Portalegre,Elvas,MT/AT,3503.74,36,208800.000,12,1207,120700
4,2026-01,2026,1,202601,Setúbal,Sines,MT/AT,2188.33,12,10461.250,15,1513,151300


Igual que en la fase de exploración, primero se convierte la columna 'data' a tipo datetime.

In [50]:
energia_injectada_df['data'] = pd.to_datetime(energia_injectada_df['data'])
energia_injectada_df['data'].min(), energia_injectada_df['data'].max()

(Timestamp('2023-01-01 00:00:00'), Timestamp('2026-03-01 00:00:00'))

Se pasa la variable 'nivel_tensao' a categórica ordenada.

In [51]:
energia_injectada_df['nivel_tensao'].unique()

array(['BTN/BTE', 'MT/AT'], dtype=object)

In [52]:
tension_levels = ['BTN/BTE', 'MT/AT']

energia_injectada_df['nivel_tensao'] = pd.Categorical(energia_injectada_df['nivel_tensao'], categories=tension_levels, ordered=True)
energia_injectada_df['nivel_tensao'].dtype

CategoricalDtype(categories=['BTN/BTE', 'MT/AT'], ordered=True, categories_dtype=object)

Los outliers se gestionan en el apartado dedicado al EDA.

Los NaN a nivel de distrito se gestionarán igual que a nivel de con_name, imputando valores a partir de sus respectivos códigos.

Para ello hace falta crear un 'map' tanto de códigos de freguesía como de concelho con sus respectivos noombres.

Primero se comprueba el formato de los códigos de distrito para todos valores missing de distrito.

In [53]:
energia_injectada_df[energia_injectada_df['distrito'].isna()]['codigo_distrito'].unique()

array(['12----', '10----', '17----', '18----', '04----', '05----', 'PT',
       '02----', '08----', '14----', '09----', '07----', '16----',
       '06----', '15----', '03----'], dtype=object)

Observamos que para los casos en los que tenemos NaN en 'distrito', los códigos de distrito asociados existen pero presentan un formato extraño, comprobaremos si difiere del formato presentado por registros sin NaN para el distrito.

In [54]:
energia_injectada_df['codigo_distrito'].unique()

array(['16', '18', '12', '15', '03', '06', '17', '14', '04', '12----',
       '02', '07', '11', '01', '09', '08', '05', '13', '10', '10----',
       '17----', '18----', '04----', '05----', 'PT', '02----', '08----',
       '14----', '09----', '07----', '16----', '06----', '15----',
       '03----'], dtype=object)

Para poder crear un mapeo correcto, hay que normalizar estos valores, incluyendo el de 'PT' que se dejará como NaN al no pertenecer a esta categoría.

In [55]:
def normalize_cod_distrito(series: pd.Series) -> pd.Series:
    return(series.astype('string')
           .str.strip()
           .str.replace('----',"")
           .replace('PT', pd.NA) #sustituye el valor entero, no necesitamos entrar al texto con '.str'
           .str.zfill(2)
           )

In [56]:
energia_injectada_df['codigo_distrito'] = normalize_cod_distrito(energia_injectada_df['codigo_distrito'])

energia_injectada_df['codigo_distrito'].unique()

<StringArray>
['16', '18', '12', '15', '03', '06', '17', '14', '04', '02', '07', '11', '01',
 '09', '08', '05', '13', '10', <NA>]
Length: 19, dtype: string

Ahora se puede crear el mapa para poder imputar valores missing de distrito. Pero antes revisamos si cada código de distrito está asociado a un solo nombre de distrito.

In [57]:
check_distrito = (energia_injectada_df.dropna(subset=['codigo_distrito', 'distrito']) # No queremos relaciones con un NaN por lado
                  .drop_duplicates(subset=['codigo_distrito', 'distrito'])) # Parejas únicas de codigo/distrito

check_distrito[check_distrito.duplicated(subset=['codigo_distrito'], keep=False)] # keep=False para dejar sin marcar los False (no duplicados) y marcar/mostrar los que si coinciden

,data,ano,mes,ano_mes,distrito,con_name,nivel_tensao,potencia_instalada,numero_de_instalacoes,energia,codigo_distrito,codigo_concelho,codigo_freguesia


No existe ningún código de distrito con múltiples valores de distrito, por lo que se crea el 'map'.

In [58]:
distrito_map = (energia_injectada_df.dropna(subset=['codigo_distrito', 'distrito'])
                .drop_duplicates(subset=['codigo_distrito'])
                .set_index(keys=['codigo_distrito'])['distrito'])
distrito_map

codigo_distrito
16    Viana do Castelo
18               Viseu
12          Portalegre
15             Setúbal
03               Braga
06             Coimbra
17           Vila Real
14            Santarém
04            Bragança
02                Beja
07               Évora
11              Lisboa
01              Aveiro
09              Guarda
08                Faro
05      Castelo Branco
13               Porto
10              Leiria
Name: distrito, dtype: object

In [59]:
mapped_distrito = pd.Series(energia_injectada_df.set_index('codigo_distrito').index.map(distrito_map), index=energia_injectada_df.index)
mapped_distrito

0        Viana do Castelo
1                   Viseu
2              Portalegre
3              Portalegre
4                 Setúbal
               ...       
20401               Viseu
20402    Viana do Castelo
20403              Lisboa
20404          Portalegre
20405               Évora
Name: codigo_distrito, Length: 20406, dtype: object

Se observan 39 valores para los cuales el codigo de distrito normalizado no encaja con ningún código del mapa creado.
Sabiendo que de entrada no existían valores NaN para códigos de distrito, muy probablemente sean aquellos registros que para distrito presentaban NaN y para código de distrito presentaban 'PT' que ha pasado a ser valor nulo al normalizar.

Por otro lado, más de 300 valores de distrito que antes eran NaN, ahora han podido ser determinados gracias al valor de su código de distrito. Se imputaran para dejar la variable preparada.

In [60]:
energia_injectada_df['distrito'] = energia_injectada_df['distrito'].fillna(mapped_distrito)
energia_injectada_df['distrito'].isna().sum()

np.int64(39)

Ahora se imputarán los valores de concelho y código de concelho en la medida de lo posible.

Lo primero es comprobar el formato de los códigos de concelho.

In [61]:
energia_injectada_df['codigo_concelho'].unique()

array(['1610', '1824', '1207', '1513', '0307', '0616', '1707', '1804',
       '1404', '1601', '0410', '1503', '12----', '1703', '0207', '0311',
       '0701', '1102', '0108', '0209', '1504', '1805', '1116', '1418',
       '0911', '0805', '0813', '1403', '0611', '1605', '1405', '0614',
       '1104', '1510', '0709', '0208', '0504', '1112', '1712', '0313',
       '0205', '1308', '1315', '1809', '0703', '1004', '0305', '1215',
       '0712', '0103', '0511', '0506', '0713', '0111', '0505', '0301',
       '0406', '1713', '1806', '1811', '0409', '1114', '1705', '0702',
       '0604', '1421', '1501', '0204', '1204', '1505', '1609', '1816',
       '1419', '1706', '0914', '0912', '0815', '10----', '0706', '0310',
       '0308', '1511', '1506', '1007', '0405', '1508', '1710', '1814',
       '0802', '0806', '0118', '0905', '1801', '0910', '0503', '0816',
       '1113', '1822', '1304', '0704', '1016', '0602', '0808', '1015',
       '1009', '1214', '1704', '0605', '0708', '0904', '0101', '0303',
  

Observamos muchos casos en los que aparecen guiones para completar el código.
Para ver la viabilidad de su imputación, observamos estos casos con detenimiento.

In [62]:
energia_injectada_df[
    energia_injectada_df["codigo_concelho"].str.contains(
        "----", na=False
    )
][["codigo_distrito", "codigo_concelho", "distrito", "con_name"]]

,codigo_distrito,codigo_concelho,distrito,con_name
17,12,12----,Portalegre,NaN
94,10,10----,Leiria,NaN
247,17,17----,Vila Real,NaN
259,10,10----,Leiria,NaN
282,17,17----,Vila Real,NaN
...,...,...,...,...
20189,16,16----,Viana do Castelo,NaN
20244,08,08----,Faro,NaN
20267,05,05----,Castelo Branco,NaN
20359,06,06----,Coimbra,NaN


Parece que estos registros no presentan información a nivel de concelho, ni en cuanto a nombre ni a código válido.

Se observan 329 casos en los que codigo_concelho presenta un formato incompleto del tipo XX---- y, simultáneamente, con_name es NaN. Además, existen 39 registros en los que codigo_concelho ya es directamente nulo.

La suma de ambos grupos, 368 registros, coincide exactamente con los 368 valores faltantes de con_name. Esto sugiere que los registros sin nombre de concelho tampoco disponen de un código de concelho válido que permita recuperarlo.

Por tanto, los códigos de concelho incompletos se convertirán a NaN, manteniéndose estos registros sin imputar a nivel de concelho. A diferencia de los nombres de distrito donde la normalización de códigos de distrito permitió la imputación, en este caso hay falta de información clave para completar los códigos de concelho afectados.

En este mismo paso se renombra 'codigo_concelho' al header común 'cod_concelho'.

In [63]:
energia_injectada_df = energia_injectada_df.rename(columns={'codigo_concelho': 'cod_concelho'}) # Header común para el código de concelho en todos los datasets
energia_injectada_df['cod_concelho'] = energia_injectada_df['cod_concelho'].astype('string').replace(r"\d{2}----", pd.NA, regex=True) # no usamos '.str' dado que reemplazamos entero

energia_injectada_df[
    ['cod_concelho', 'con_name']
].isna().sum()

cod_concelho    368
con_name        368
dtype: int64

Se hacen comprobaciones textuales:

In [64]:
energia_injectada_df['nivel_tensao'].value_counts(dropna=False)

nivel_tensao
BTN/BTE    10842
MT/AT       9564
Name: count, dtype: int64

In [65]:
distrito_check = (
    energia_injectada_df
    .dropna(subset=['distrito'])
    .assign(normalized=energia_injectada_df['distrito'].astype('string').str.strip().str.lower())
    .groupby('normalized')['distrito']
    .nunique()
)

distrito_check[distrito_check > 1]

Series([], Name: distrito, dtype: int64)

In [66]:
concelho_check = (
    energia_injectada_df
    .dropna(subset=['con_name'])
    .assign(normalized=energia_injectada_df['con_name'].astype('string').str.strip().str.lower())
    .groupby('normalized')['con_name']
    .nunique()
)

concelho_check[concelho_check > 1]

Series([], Name: con_name, dtype: int64)

In [67]:
energia_injectada_df.groupby('codigo_distrito')['distrito'].nunique().sort_values(ascending=False).max()

np.int64(1)

In [68]:
energia_injectada_df.groupby('cod_concelho')['con_name'].nunique().sort_values(ascending=False).max()

np.int64(1)

In [69]:
energia_injectada_df.groupby('con_name')['cod_concelho'].nunique().max()

np.int64(1)

Como en el resto de datasets, esta comprobación valida la consistencia de la fuente; los nombres definitivos se asignarán desde la tabla de referencia geográfica a partir de 'cod_concelho' (ver nota en '8-total-upac-mensal').

Comprobación final del dataset entero:

In [70]:
energia_injectada_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20406 entries, 0 to 20405
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   data                   20406 non-null  datetime64[ns]
 1   ano                    20406 non-null  int64         
 2   mes                    20406 non-null  int64         
 3   ano_mes                20406 non-null  int64         
 4   distrito               20367 non-null  object        
 5   con_name               20038 non-null  object        
 6   nivel_tensao           20406 non-null  category      
 7   potencia_instalada     20406 non-null  float64       
 8   numero_de_instalacoes  20406 non-null  int64         
 9   energia                20406 non-null  float64       
 10  codigo_distrito        20367 non-null  string        
 11  cod_concelho           20038 non-null  string        
 12  codigo_freguesia       20406 non-null  object        
dtypes

In [71]:
energia_injectada_df.to_parquet(r"../data_interim/energia_injectada.parquet", index=False)

## **26-centrais**: Flujo de nuevas unidades de autoconsumo conectadas

In [72]:
centrais_df = pd.read_csv(r"../data_raw/26-centrais.csv", sep=';', encoding='utf-8')
centrais_df.head(5)

,ano,mes,con_name,cod_concelho,potencia_ligacao,processos_concluidos_nr
0,2024,11,Albufeira,801,10.96,3
1,2024,11,Elvas,1207,10.00,1
2,2024,11,Seixal,1510,15.57,6
3,2024,11,Loulé,808,110.00,1
4,2024,11,Setúbal,1512,12.00,1


Se crea la feature 'data' y se normalizan los códigos de concelho. En este dataset la columna ya se llama 'cod_concelho' (header común), por lo que no hace falta renombrarla.

In [73]:
centrais_df['data'] = pd.to_datetime(centrais_df[['ano', 'mes']].rename(columns={'ano':'year', 'mes':'month'}).assign(day=1))
centrais_df['data'].dtype

dtype('<M8[ns]')

In [74]:
centrais_df['cod_concelho'] = normalize_cod_concelho(centrais_df['cod_concelho'])

A diferencia de los datasets anteriores, se comprueba que exista de entrada un solo nombre de concelho por código.

In [75]:
centrais_con_check = (centrais_df
                      .dropna(subset=['con_name'])
                      .assign(normalized = centrais_df['con_name'].astype('string').str.strip().str.lower())
                      .groupby('normalized')['con_name']
                      .nunique())

centrais_con_check[centrais_con_check>1]

Series([], Name: con_name, dtype: int64)

Solo existe un nombre de igual formato para cada concelho.

Se comprueba entonces que cada uno de ellos esté asociado a un máximo de un código

In [76]:
centrais_df.groupby('con_name')['cod_concelho'].nunique().max()

np.int64(1)

In [77]:
centrais_df.groupby('cod_concelho')['con_name'].nunique().max()

np.int64(1)

Vemos que hay 1 código de concelho para cada nombre de concelho y viceversa.

Parece que no hay valores nulos ni para cod_concelho ni para con_name, de todos modos comprobaremos que no haya valores extraños.

Como en el resto de datasets, esta comprobación valida la consistencia de la fuente; los nombres definitivos se asignarán desde la tabla de referencia geográfica a partir de 'cod_concelho' (ver nota en '8-total-upac-mensal').

In [78]:
centrais_df['cod_concelho'].unique()

<StringArray>
['0801', '1207', '1510', '0808', '1512', '1414', '1714', '0705', '1103',
 '0306',
 ...
 '0802', '1819', '0207', '0506', '0112', '1202', '1211', '0708', '1704',
 '0908']
Length: 278, dtype: string

In [79]:
centrais_df['con_name'].unique()

array(['Albufeira', 'Elvas', 'Seixal', 'Loulé', 'Setúbal', 'Rio Maior',
       'Vila Real', 'Évora', 'Azambuja', 'Esposende', 'Ponte de Sor',
       'Loures', 'Ponte de Lima', 'Tavira', 'Sesimbra', 'Porto',
       'Abrantes', 'Alcanena', 'Montemor-o-Velho', 'Valongo', 'Lagos',
       'Arganil', 'Montijo', 'Oliveira do Hospital', 'Mira', 'Vila Verde',
       'Viana do Castelo', 'Montemor-o-Novo', 'Tomar', 'Almeirim',
       'Leiria', 'Grândola', 'Oliveira do Bairro', 'Miranda do Corvo',
       'Lourinhã', 'Santo Tirso', 'Paços de Ferreira', 'Mangualde',
       'Pombal', 'Alcobaça', 'Carregal do Sal', 'Coimbra', 'Trofa',
       'Paredes', 'Odivelas', 'Santiago do Cacém', 'Cantanhede',
       'Marco de Canaveses', 'Baião', 'Soure', 'São Brás de Alportel',
       'Vila Pouca de Aguiar', 'Mértola', 'Penedono', 'Braga',
       'Arcos de Valdevez', 'Portalegre', 'Odemira', 'Lousada',
       'Vendas Novas', 'Peso da Régua', 'Castelo Branco', 'Vimioso',
       'Vila Real de Santo António', 'Pal

Todo correcto, por lo que finalmente se procede a la gestión de valores nulos relativos a 'potencia_ligacao', dado que el resto de variables parecen ser correctas.

Primero se observan muestras de los 796 nulos que se presentan.

In [80]:
centrais_df[centrais_df['potencia_ligacao'].isna()].head(10)

,ano,mes,con_name,cod_concelho,potencia_ligacao,processos_concluidos_nr,data
19,2024,11,Porto,1312,NaN,2,2024-11-01
315,2024,12,Loures,1107,NaN,1,2024-12-01
374,2024,12,Cascais,1105,NaN,1,2024-12-01
492,2024,12,Vila Nova de Gaia,1317,NaN,13,2024-12-01
574,2024,12,Vila Franca de Xira,1114,NaN,1,2024-12-01
729,2025,1,Barcelos,0302,NaN,1,2025-01-01
872,2025,1,Monção,1604,NaN,1,2025-01-01
947,2025,1,Trofa,1318,NaN,1,2025-01-01
952,2025,1,Castelo de Vide,1205,NaN,4,2025-01-01
963,2025,1,Setúbal,1512,NaN,4,2025-01-01


Se observa que aunque la potencia instalada es nula, si que hay valores para procesos concluídos, habrá que observar un caso entero para determinar si es potencialmente un valor clave para imputar los 'missing'.

In [81]:
centrais_df[(centrais_df['cod_concelho'] == '1312') & (centrais_df['processos_concluidos_nr']==2) & (centrais_df['mes']==11)]

,ano,mes,con_name,cod_concelho,potencia_ligacao,processos_concluidos_nr,data
19,2024,11,Porto,1312,NaN,2,2024-11-01
13198,2024,11,Porto,1312,2.73,2,2024-11-01
15348,2021,11,Porto,1312,2.30,2,2021-11-01
15485,2021,11,Porto,1312,4.00,2,2021-11-01
21110,2025,11,Porto,1312,4.76,2,2025-11-01
21112,2025,11,Porto,1312,83.00,2,2025-11-01
26689,2023,11,Porto,1312,20.00,2,2023-11-01
26831,2023,11,Porto,1312,2.47,2,2023-11-01
40324,2025,11,Porto,1312,8.00,2,2025-11-01
40451,2025,11,Porto,1312,3.00,2,2025-11-01


No parece que la potencia instalada dependa del número de procesos ni se mantenga constante para un mismo mes, por lo que cabe concluir que los valores nulos no seran imputables.

Se comprueba y guarda el dataset interim.

In [82]:
centrais_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77930 entries, 0 to 77929
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ano                      77930 non-null  int64         
 1   mes                      77930 non-null  int64         
 2   con_name                 77930 non-null  object        
 3   cod_concelho             77930 non-null  string        
 4   potencia_ligacao         77134 non-null  float64       
 5   processos_concluidos_nr  77930 non-null  int64         
 6   data                     77930 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(3), object(1), string(1)
memory usage: 4.2+ MB


In [83]:
centrais_df.to_parquet(r"../data_interim/26-centrais.parquet", index=False)

## **comunidades-de-energia**: Número de comunidades de energias activas por tipo

In [84]:
comunidades_df = pd.read_csv(r"../data_raw/comunidades-de-energia.csv", sep=';', encoding='utf-8')
comunidades_df.head(5)

,data,ano,mes,dis_name,con_name,fre_name,tipo_acc_cer,contagem_tipo_acc_cer,codigo_distrito,codigo_concelho,codigo_freguesia
0,2025-09,2025,9,Braga,Braga,Adaúfe,ACC,3,3,303,030301
1,2025-09,2025,9,Porto,Santo Tirso,Água Longa,ACC,1,13,1314,131402
2,2025-09,2025,9,Lisboa,Amadora,Águas Livres,ACC,1,11,1115,111513
3,2025-09,2025,9,Santarém,Santarém,Alcanede,ACC,1,14,1416,141604
4,2025-09,2025,9,Lisboa,Lisboa,Alcântara,ACC,1,11,1106,110602


Tal como sucede con el resto de datasets, hay que transformar la columna 'data' a datetime.

In [85]:
comunidades_df['data'] = pd.to_datetime(comunidades_df['data'])
comunidades_df['data'].dtype

dtype('<M8[ns]')

En relación a los códigos de concelho, se renombran al header común 'cod_concelho', se normalizan y posteriormente se comprueba que cada uno se relacione con un solo nombre de concelho.

In [86]:
comunidades_df['codigo_concelho'].unique()

array([ 303, 1314, 1115, 1416, 1106, 1414,  803,  605,  606, 1807,  307,
        702, 1102, 1207, 1421, 1316, 1412, 1304, 1004, 1307,  308, 1107,
        112, 1809,  705, 1317,  610, 1306, 1610, 1510,  711, 1309, 1315,
        115, 1001,  613,  313,  110, 1821,  312,  502, 1305, 1419,  109,
        609, 1415,  114, 1012,  107, 1310,  213,  302,  309,  808, 1111,
        505, 1713,  402, 1405,  314,  603, 1204, 1418,  113, 1507, 1215,
       1402, 1312,  104,  205, 1005, 1318,  305, 1104,  602, 1105, 1503,
        604, 1409,  504, 1015, 1009,  706, 1110, 1308, 1708, 1116,  508,
        304, 1214,  911, 1512, 1401, 1509, 1820, 1006,  101, 1303, 1601,
       1805,  712,  816, 1823, 1406, 1506,  103, 1403,  306, 1714,  406,
       1609, 1114, 1213, 1103, 1113,  616,  102, 1109,  105, 1410, 1404,
        907, 1413, 1508, 1014, 1511,  108, 1504, 1203,  407, 1311, 1112,
       1301, 1101, 1603, 1205, 1212, 1313,  510, 1817,  201, 1010,  814,
       1505, 1008,  806, 1816,  703, 1604,  813,  8

In [87]:
comunidades_df = comunidades_df.rename(columns={'codigo_concelho': 'cod_concelho'}) # Header común para el código de concelho en todos los datasets
comunidades_df['cod_concelho'] = normalize_cod_concelho(comunidades_df['cod_concelho'])
comunidades_df['cod_concelho'].unique()

<StringArray>
['0303', '1314', '1115', '1416', '1106', '1414', '0803', '0605', '0606',
 '1807',
 ...
 '1003', '0209', '1201', '0617', '0601', '0913', '1002', '0815', '1818',
 '0404']
Length: 251, dtype: string

In [88]:
(comunidades_df
 .dropna(subset=['con_name'])
 .assign(normalized = comunidades_df['con_name'].astype('string').str.strip().str.lower())
 .groupby('normalized')['con_name']
 .nunique()
).max()

np.int64(1)

In [89]:
comunidades_df.groupby('cod_concelho')['con_name'].nunique().max()

np.int64(1)

In [90]:
comunidades_df.groupby('con_name')['cod_concelho'].nunique().max()

np.int64(1)

Se ha normalizado el código de concelho y se ha comprobado que solo existe un nombre de concelho por código. Como en el resto de datasets, esta comprobación valida la consistencia de la fuente; los nombres definitivos se asignarán desde la tabla de referencia geográfica a partir de 'cod_concelho' (ver nota en '8-total-upac-mensal').
No hay necesidad de gestionar los nulos por freguesia.

En relación a las variables 'tipo_acc_cer' y 'contagem_tipo_acc_cer' se comprueban tipos y de ser necesario se transforman en categorical e integer respectivamente. En el caso de 'tipo_acc_cer' no presenta orden en sus categorías.

In [91]:
comunidades_df['tipo_acc_cer'].dtype

dtype('O')

In [92]:
comunidades_df['contagem_tipo_acc_cer'].dtype

dtype('int64')

In [93]:
comunidades_df['tipo_acc_cer'] = comunidades_df['tipo_acc_cer'].astype('category')
comunidades_df['tipo_acc_cer'].dtype

CategoricalDtype(categories=['ACC', 'CER'], ordered=False, categories_dtype=object)

Vemos que hay dos categrorías de comunidades, ACC para Autoconsumo colectivo y CER para comunidades de energía renovable.

ACC es un modelo enfocado estrictamente en la producción y consumo compartido de energía electrica entre varios consumidores asociados a una o varias instalaciones de generación (generalmente fotovoltaica) en un mismo edificio o urbanización.

Una comunidad de energía renovable es una estructura jurídica más ámplia y autónoma (asociación, cooperativa, etc.) que además de producir y consumir energía renovable, también getstiona almacenamientro o comercializa excedentes.
Se centran en fomentar la transición energética local, cohesión social y lucha contra la pobreza energética integrando ciudadanos, pymes, autoridades locales, etc.

La gestión de los NaN restantes, outliers, agrupación por concelho, etc. se realizará en el EDA; por ahora se guarda la versión modificada del dataset.

In [94]:
comunidades_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11007 entries, 0 to 11006
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   data                   11007 non-null  datetime64[ns]
 1   ano                    11007 non-null  int64         
 2   mes                    11007 non-null  int64         
 3   dis_name               11007 non-null  object        
 4   con_name               11007 non-null  object        
 5   fre_name               10823 non-null  object        
 6   tipo_acc_cer           11007 non-null  category      
 7   contagem_tipo_acc_cer  11007 non-null  int64         
 8   codigo_distrito        11007 non-null  int64         
 9   cod_concelho           11007 non-null  string        
 10  codigo_freguesia       11007 non-null  object        
dtypes: category(1), datetime64[ns](1), int64(4), object(4), string(1)
memory usage: 870.9+ KB


In [95]:
comunidades_df.to_parquet(r"../data_interim/comunidades_energia.parquet", index=False)

## **codigo-de-atividade-economica-por-distrito-concelho-e-freguesia**: Actividad economica por freguesia

In [96]:
atividade_economica_df = pd.read_csv(r"../data_raw/codigo-de-atividade-economica-por-distrito-concelho-e-freguesia.csv", sep=';', encoding='utf-8')
atividade_economica_df.head(5)

,ano,nuts_iii,codigo_cae,cae,no_de_empresas,dis_name,con_name,freguesia,codigo_nuts_iii,codigo_distrito,codigo_concelho,codigo_freguesia
0,2023,Alto Minho,F,Construção,6,Viana do Castelo,Arcos de Valdevez,Aboim das Choças,111,16,1601,160101
1,2023,Alto Minho,G,Comércio por grosso e a retalho; reparação de ...,7,Viana do Castelo,Arcos de Valdevez,Aboim das Choças,111,16,1601,160101
2,2023,Alto Minho,I,"Alojamento, restauração e similares",3,Viana do Castelo,Arcos de Valdevez,Aboim das Choças,111,16,1601,160101
3,2023,Alto Minho,P,Educação,3,Viana do Castelo,Arcos de Valdevez,Aboim das Choças,111,16,1601,160101
4,2023,Alto Minho,Q,Actividades de saúde humana e apoio social,5,Viana do Castelo,Arcos de Valdevez,Aboim das Choças,111,16,1601,160101


Similar al caso anterior, no gestionaremos los NaN de freguesia dada su nula incidencia en el proyecto, por otro lado, si que se transforman tipos de datos y se comprueba presencia única de un solo nombre y formato de concelho (clave para agrupación) así como un formato correcto para el resto de métricas del dataset.

En este dataset no existe granularidad mensual. Sin embargo, los datos anuales se consideran constantes a lo largo del año indicado, ya que no se producen cambios en la actividad económica. Por tanto, la idea general será realizar un 'join' con un dataset que sí incluya la variable mensual, de forma que, para cada mes de un mismo año y región, se asigne la misma actividad económica (CAE) mediante propagación.

En relación al tipo de dato de 'año' se queda como 'int64' para mantener homogeneidad entre datasets de cara a hacer el futuro merge.

In [97]:
atividade_economica_df['codigo_cae'].unique()

array(['F', 'G', 'I', 'P', 'Q', 'A', 'D', 'S', 'H', 'L', 'N', 'R', 'C',
       'M', 'E', 'J', 'B'], dtype=object)

In [98]:
atividade_economica_df['cae'].unique()

array(['Construção',
       'Comércio por grosso e a retalho; reparação de veículos automóveis e motociclos',
       'Alojamento, restauração e similares', 'Educação',
       'Actividades de saúde humana e apoio  social',
       'Agricultura, produção animal, caça,  floresta e pesca',
       'Electricidade, gás, vapor,  água quente e fria e ar frio',
       'Outras actividades de serviços', 'Transportes e armazenagem',
       'Actividades imobiliárias',
       'Actividades administrativas e dos serviços de apoio',
       'Actividades artísticas, de espectáculos, desportivas e recreativas',
       'Indústrias transformadoras',
       'Actividades de consultoria,  científicas, técnicas e similares',
       'Captação, tratamento e distribuição de água; saneamento,  gestão de resíduos e despoluição',
       'Actividades de informação e de comunicação',
       'Indústrias extractivas'], dtype=object)

En este caso, al haber pocas categorías no hace falta comprobar si hay valores mal formatados o repetidos, simplemente se transforman los tipos de datos a categóricos sin ordenar para ambos casos y se procede normalizar los códigos de concelho.

Para el número de empresass por región, dado que ya se interpretan como 'integers' y no presentan valores nulos, no habrá que realizar transformaciones.

In [99]:
atividade_economica_df['codigo_cae'] = atividade_economica_df['codigo_cae'].astype('category')
atividade_economica_df['cae'] = atividade_economica_df['cae'].astype('category')

atividade_economica_df['codigo_cae'].dtype, atividade_economica_df['cae'].dtype

(CategoricalDtype(categories=['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'L', 'M',
                   'N', 'P', 'Q', 'R', 'S'],
 , ordered=False, categories_dtype=object),
 CategoricalDtype(categories=['Actividades administrativas e dos serviços de apoio',
                   'Actividades artísticas, de espectáculos, desportivas e recreativas',
                   'Actividades de consultoria,  científicas, técnicas e similares',
                   'Actividades de informação e de comunicação',
                   'Actividades de saúde humana e apoio  social',
                   'Actividades imobiliárias',
                   'Agricultura, produção animal, caça,  floresta e pesca',
                   'Alojamento, restauração e similares',
                   'Captação, tratamento e distribuição de água; saneamento,  gestão de resíduos e despoluição',
                   'Comércio por grosso e a retalho; reparação de veículos automóveis e motociclos',
                   'Construção', 'Ed

Se renombra el código de concelho al header común 'cod_concelho', se normaliza y se comprueba que para cada nombre de concelho hay un solo formato.

In [100]:
atividade_economica_df = atividade_economica_df.rename(columns={'codigo_concelho': 'cod_concelho'}) # Header común para el código de concelho en todos los datasets
atividade_economica_df['cod_concelho'] = normalize_cod_concelho(atividade_economica_df['cod_concelho'])
atividade_economica_df['cod_concelho'].dtype

string[python]

In [101]:
(atividade_economica_df.dropna(subset=['con_name'])
 .assign(normalized = atividade_economica_df['con_name'].astype('string').str.strip().str.lower())
 .groupby('normalized')['con_name']
 .nunique()
 ).max()

np.int64(1)

Se procede a comprobar que existe un solo código de concelho asociado a un nombre de concelho y viceversa.

In [102]:
atividade_economica_df.groupby('cod_concelho')['con_name'].nunique().max()

np.int64(1)

In [103]:
atividade_economica_df.groupby('con_name')['cod_concelho'].nunique().max()

np.int64(2)

Por primera vez se encuentra más de un código de concelho para un mismo nombre. Hay que investigar si es una inconsistencia o no.

In [104]:
check = (
    atividade_economica_df
    .groupby('con_name')['cod_concelho']
    .nunique()
)

check[check > 1]

con_name
Lagoa    2
Name: cod_concelho, dtype: int64

In [105]:
conflict_names = check[check > 1].index

atividade_economica_df[
    atividade_economica_df['con_name'].isin(conflict_names)
][['dis_name', 'codigo_distrito','con_name', 'cod_concelho']].drop_duplicates()

,dis_name,codigo_distrito,con_name,cod_concelho
12170,Faro,8,Lagoa,0806
12560,NaN,42,Lagoa,4201


In [106]:
total_upac_df[
    total_upac_df['concelho'].eq('Lagoa')
][['distrito', 'concelho', 'cod_concelho']].drop_duplicates()

,distrito,concelho,cod_concelho
380,Faro,Lagoa,0806


Se comprueba que, a diferencia de los anteriores datasets (p.ej. 'total_upac'), en el dataset de actividad economica se tienen en cuenta las Açores, mientras que en el resto solo se considera Portugal continental.

En este caso, el NaN corresponde a 'Açores', aunque de todos modos, dado que el estudio se hará únicamente de Portugal continental, no hay necesidad de gestionar esos valores nulos, pues al combinar datasets quedaran excluídos de forma automática.

Este caso (dos concelhos llamados 'Lagoa', uno en el Algarve y otro en Açores) muestra por qué los datasets nunca se cruzarán por nombre, sino por 'cod_concelho'. Los nombres definitivos se asignarán desde la tabla de referencia geográfica, donde INE los distingue (p.ej. 'Lagoa (R.A.A.)'). Ver 'app/utils/concelho_codes/README.md'.

In [107]:
atividade_economica_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39686 entries, 0 to 39685
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   ano               39686 non-null  int64   
 1   nuts_iii          39686 non-null  object  
 2   codigo_cae        39686 non-null  category
 3   cae               39686 non-null  category
 4   no_de_empresas    39686 non-null  int64   
 5   dis_name          36981 non-null  object  
 6   con_name          39686 non-null  object  
 7   freguesia         39686 non-null  object  
 8   codigo_nuts_iii   39686 non-null  object  
 9   codigo_distrito   39686 non-null  int64   
 10  cod_concelho      39686 non-null  string  
 11  codigo_freguesia  39686 non-null  object  
dtypes: category(2), int64(3), object(6), string(1)
memory usage: 3.1+ MB


In [108]:
atividade_economica_df.to_parquet(r"../data_interim/atividade_economica.parquet", index=False)

## **clientes-por-escalao-de-potencia**: Número de contratos por segmento de potencia contratada

In [109]:
clientes_escalao_df = pd.read_csv(r"../data_raw/clientes-por-escalao-de-potencia.csv", sep=';', encoding='utf-8')
clientes_escalao_df.head(5)

,data,ano,mes,dis_name,con_name,fre_name,segmento_de_potencia_contratada,numero_de_contratos,dis_code,con_code,fre_code
0,2025-09,2025,9,Évora,Évora,Nossa Senhora da Tourega e Nossa Senhora de Gu...,]20 30],2.0,7,705,070525
1,2025-09,2025,9,Évora,Évora,Nossa Senhora da Tourega e Nossa Senhora de Gu...,]40 50],15.0,7,705,070525
2,2025-09,2025,9,Évora,Évora,Nossa Senhora da Tourega e Nossa Senhora de Gu...,]50 100],6.0,7,705,070525
3,2025-09,2025,9,Évora,Évora,Nossa Senhora da Tourega e Nossa Senhora de Gu...,]250 300],2.0,7,705,070525
4,2025-09,2025,9,Évora,Évora,Nossa Senhora da Tourega e Nossa Senhora de Gu...,]400 450],1.0,7,705,070525


Lo primero es transformar 'data' a datetime, renombrar el código de concelho ('con_code') al header común 'cod_concelho' y normalizarlo.

In [110]:
clientes_escalao_df['data'] = pd.to_datetime(clientes_escalao_df['data'])
clientes_escalao_df['data'].dtype

dtype('<M8[ns]')

In [111]:
clientes_escalao_df = clientes_escalao_df.rename(columns={'con_code': 'cod_concelho'}) # Header común para el código de concelho en todos los datasets
clientes_escalao_df['cod_concelho'] = normalize_cod_concelho(clientes_escalao_df['cod_concelho'])
clientes_escalao_df['cod_concelho'].unique()

<StringArray>
['0705', '0706', '0707', '0708', '0709', '0710', '0711', '0712', '0713',
 '0714',
 ...
 '1806', '1807', '1808', '1809', '1810', '1811', '1812', '1813', '1814',
 '1815']
Length: 278, dtype: string

Se ha comprobado en la fase de inspección el hecho de que los números de contratos, a pesar de ser considerados como 'float64', pueden y deben ser convertidos a 'int64'.

In [112]:
(clientes_escalao_df['numero_de_contratos'] % 1 == 0).all()

np.True_

In [113]:
clientes_escalao_df['numero_de_contratos'] = clientes_escalao_df['numero_de_contratos'].astype('int')
clientes_escalao_df['numero_de_contratos'].dtype

dtype('int64')

También hay que convertir 'segmento_de_potencia_contratada' a categórico con orden.
Para eso habrá que gestionar los valores no atribuídos que aún no son nulos y aquellos valores numéricos que no se integran dentro de una de las categorías existentes.

In [114]:
clientes_escalao_df['segmento_de_potencia_contratada'].unique()

array([']20 30]', ']40 50]', ']50 100]', ']250 300]', ']400 450]', '6.9',
       '17.25', '20.7', '27.6', '34.5', ']1500 2000]', '5.75', '41.4',
       '4.6', ']10 20]', ']150 200]', '1.15', '10.35', '2.3', ']0 10]',
       ']100 150]', ']30 40]', '3.45', ']500 1000]', '13.8', ']300 350]',
       ']350 400]', ']200 250]', ']10000 15000]', ']1000 1500]',
       ']450 500]', ']4000 4500]', ']3000 3500]', ']5000 10000]',
       ']2000 2500]', ']2500 3000]', ']15000 20000]', ']4500 5000]',
       ']3500 4000]', ']20000 25000]', ']35000 40000]', ']25000 30000]',
       ']40000 45000]', ']45000 50000]', '>50000', ']30000 35000]',
       'Não atribuído'], dtype=object)

Antes de nada, 'Nao Atribuído' pasa a ser 'NaN', al no tener valor asignado

In [115]:
clientes_escalao_df['segmento_de_potencia_contratada'] = clientes_escalao_df['segmento_de_potencia_contratada'].replace('Não atribuído', pd.NA)

In [116]:
bins = [
    0, 10, 20, 30, 40, 50, 100, 150, 200, 250, 300,
    350, 400, 450, 500, 1000, 1500, 2000, 2500, 3000,
    3500, 4000, 4500, 5000, 10000, 15000, 20000, 25000,
    30000, 35000, 40000, 45000, 50000, np.inf
]

labels = [
    ']0 10]', ']10 20]', ']20 30]', ']30 40]', ']40 50]',
    ']50 100]', ']100 150]', ']150 200]', ']200 250]',
    ']250 300]', ']300 350]', ']350 400]', ']400 450]',
    ']450 500]', ']500 1000]', ']1000 1500]', ']1500 2000]',
    ']2000 2500]', ']2500 3000]', ']3000 3500]', ']3500 4000]',
    ']4000 4500]', ']4500 5000]', ']5000 10000]',
    ']10000 15000]', ']15000 20000]', ']20000 25000]',
    ']25000 30000]', ']30000 35000]', ']35000 40000]',
    ']40000 45000]', ']45000 50000]', '>50000'
]

numeric_values = pd.to_numeric(
    clientes_escalao_df['segmento_de_potencia_contratada'],
    errors='coerce'
)

numeric_mask = numeric_values.notna()

clientes_escalao_df.loc[
    numeric_mask,
    'segmento_de_potencia_contratada'
] = pd.cut(
    numeric_values[numeric_mask],
    bins=bins,
    labels=labels,
    right=True
).astype('string')

clientes_escalao_df['segmento_de_potencia_contratada'] = pd.Categorical(
    clientes_escalao_df['segmento_de_potencia_contratada'],
    categories=labels,
    ordered=True
)

Sacando los índices de los valores numéricos vez creados los 'bins' donde caerán los valores numéricos, la idea es crear una máscara a partir de localizar todos aquellos valores que no se transforman en NaN al pasarlas a categoría numérica.

Con los valores índices de dichos valores, mediante la función '.loc[]' se indica la columna sobre la que se quieren localizar y se sustituyen por el 'bin' pertinente y adjudicado mediante la función '.cut()', con la lista de 'bins' sabe en que corte queda el número y con labels le asigna un tramo, con right 'True' nos aseguramos incluir el límite derecho de cada intervalo. Posteriormente se guardan en formato texto para poder escribirlo en la columna original que será posteriormente transformada a categoría aprovechando los labels ya ordenados como orden de categorización.

In [117]:
clientes_escalao_df[
    'segmento_de_potencia_contratada'
].value_counts(dropna=False, sort=False)

segmento_de_potencia_contratada
]0 10]           863879
]10 20]          400397
]20 30]          262068
]30 40]           95228
]40 50]          178573
]50 100]          76921
]100 150]         62230
]150 200]         46486
]200 250]         28758
]250 300]         34594
]300 350]         16525
]350 400]         21413
]400 450]          9450
]450 500]         15784
]500 1000]        31506
]1000 1500]       15006
]1500 2000]        7773
]2000 2500]        4505
]2500 3000]        3264
]3000 3500]        2209
]3500 4000]        1425
]4000 4500]         796
]4500 5000]        1100
]5000 10000]       3585
]10000 15000]       933
]15000 20000]       386
]20000 25000]       248
]25000 30000]       231
]30000 35000]        61
]35000 40000]        74
]40000 45000]       121
]45000 50000]       104
>50000              179
NaN                  41
Name: count, dtype: int64

In [118]:
clientes_escalao_df['segmento_de_potencia_contratada'].dtype

CategoricalDtype(categories=[']0 10]', ']10 20]', ']20 30]', ']30 40]', ']40 50]',
                  ']50 100]', ']100 150]', ']150 200]', ']200 250]',
                  ']250 300]', ']300 350]', ']350 400]', ']400 450]',
                  ']450 500]', ']500 1000]', ']1000 1500]', ']1500 2000]',
                  ']2000 2500]', ']2500 3000]', ']3000 3500]', ']3500 4000]',
                  ']4000 4500]', ']4500 5000]', ']5000 10000]',
                  ']10000 15000]', ']15000 20000]', ']20000 25000]',
                  ']25000 30000]', ']30000 35000]', ']35000 40000]',
                  ']40000 45000]', ']45000 50000]', '>50000'],
, ordered=True, categories_dtype=object)

Si bien no hay NaN a gestionar, se comprueba que el formato de nombre de concelho es correcto, que cada nombre de concelho tiene un código y viceversa sin contar excepciones como el caso de las Açores.

In [119]:
(clientes_escalao_df.dropna(subset=['con_name'])
 .assign(normalized = clientes_escalao_df['con_name'].astype('string').str.strip().str.lower())
 .groupby('normalized')['con_name']
 .nunique()).max()

np.int64(1)

In [120]:
clientes_escalao_df.groupby('cod_concelho')['con_name'].nunique().max()

np.int64(1)

In [121]:
clientes_escalao_df.groupby('con_name')['cod_concelho'].nunique().max()

np.int64(1)

Como en el resto de datasets, esta comprobación valida la consistencia de la fuente; los nombres definitivos se asignarán desde la tabla de referencia geográfica a partir de 'cod_concelho' (ver nota en '8-total-upac-mensal').

Todo parece correcto, se guarda la versión modificada.

In [122]:
clientes_escalao_df.to_parquet(r"../data_interim/clientes_escalao.parquet", index=False)

## **25-plr-producao-renovavel**: Nuevas conexiones a la red de centrales de energía renovable

In [123]:
producao_renovavel_df = pd.read_csv(r"../data_raw/25-plr-producao-renovavel.csv", sep=';', encoding='utf-8')
producao_renovavel_df.head(5)

,ano,mes,con_name,cod_concelho,potencia_ligacao,pedidos_ligacao_rede_executados_nr
0,2023,12,Santarém,1416,3000,3
1,2024,8,Santarém,1416,3000,3
2,2025,3,Santarém,1416,1000,1
3,2021,6,Salvaterra de Magos,1415,25000,1
4,2025,11,Salvaterra de Magos,1415,1000,1


Se crea la columna 'data' a partir del año y mes por registro.

In [124]:
producao_renovavel_df['data'] = pd.to_datetime(producao_renovavel_df[['ano', 'mes']].rename(columns={'ano': 'year', 'mes': 'month'}).assign(day=1))
producao_renovavel_df.insert(0,'data',producao_renovavel_df.pop('data'))
producao_renovavel_df['data'].dtype

dtype('<M8[ns]')

Al no haber valores nulos, sencillamente se normaliza el código de concelho (ya se llama 'cod_concelho', el header común) y se comprueba que para cada código haya un nombre y viceversa. El resto de variables y métricas presentan valores numéricos correctos.

In [125]:
producao_renovavel_df['cod_concelho'] = normalize_cod_concelho(producao_renovavel_df['cod_concelho'])
producao_renovavel_df['cod_concelho'].unique()

<StringArray>
['1416', '1415', '0704', '0210', '0705', '1418', '0213', '0605', '0701',
 '0302',
 ...
 '1308', '1809', '1013', '0903', '0706', '0117', '0208', '0811', '0803',
 '0813']
Length: 107, dtype: string

In [126]:
(
    producao_renovavel_df
    .dropna(subset=['con_name'])
    .assign(
        normalized=producao_renovavel_df['con_name']
        .astype('string')
        .str.strip()
        .str.lower()
    )
    .groupby('normalized')['con_name']
    .nunique()
).max()

np.int64(1)

In [127]:
producao_renovavel_df.groupby('cod_concelho')['con_name'].nunique().max()

np.int64(1)

In [128]:
producao_renovavel_df.groupby('con_name')['cod_concelho'].nunique().max()

np.int64(1)

Como en el resto de datasets, esta comprobación valida la consistencia de la fuente; los nombres definitivos se asignarán desde la tabla de referencia geográfica a partir de 'cod_concelho' (ver nota en '8-total-upac-mensal').

In [129]:
producao_renovavel_df.to_parquet(r"../data_interim/producao_renovavel.parquet", index=False)